In [0]:
# https://adb-7405614744420764.4.azuredatabricks.net/marketplace/consumer/listings/ea296770-0ee0-4b74-a202-a2c0873add7c?o=7405614744420764

In [0]:
import urllib.request

# setup
url = "https://adb-7405614744420764.4.azuredatabricks.net/marketplace/consumer/listings/ea296770-0ee0-4b74-a202-a2c0873add7c?o=7405614744420764"
dest = "/dbfs/mnt/data/raw/data_from_marketplace.csv" # Це шлях через DBFS mount або скористайся abfss посиланням

urllib.request.urlretrieve(url, dest)
print("Готово!")

In [0]:
# Подивимось на список файлів у папці з тиском
pressure_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/08.Formation_Pressure_Data/"

files = dbutils.fs.ls(pressure_path)
for f in files:
    print(f"Файл: {f.name}, Розмір: {f.size / 1024 / 1024:.2f} MB")

# Давай прочитаємо шматочок першого ліпшого CSV, щоб зрозуміти структуру
# Якщо там є CSV, підстав назву нижче
sample_file = files[0].path 
if ".csv" in sample_file:
    df_sample = spark.read.option("header", "true").csv(sample_file)
    display(df_sample.limit(5))
else:
    print("Файл не в CSV форматі, спробуємо інший підхід.")

In [0]:
import pandas as pd
from datetime import datetime

# Створюємо список свердловин, які реально є в Northern Lights
wells = ["31/5-7", "31/2-1", "31/5-5"] 

sap_data = []
for well in wells:
    sap_data.append({
        "well_id": well,
        "cost_center": "CC_NORTH_SEA_001",
        "op_cost_per_day": 15000.0,
        "currency": "USD",
        "asset_manager": "A. Pandey", # Маленька пасхалка для інтерв'юера
        "last_update": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

# Перетворюємо в Spark DataFrame
sap_df = spark.createDataFrame(sap_data)

# Записуємо на ADLS у папку landing (імітуємо вивантаження з SAP)
sap_storage_path = "abfss://data@stshellcrude.dfs.core.windows.net/landing/sap_export/"
sap_df.write.format("csv").option("header", "true").mode("overwrite").save(sap_storage_path)

print(f"SAP-like дані успішно згенеровані та завантажені в Landing Zone: {sap_storage_path}")

In [0]:
# Path to the ASCII info file we found earlier
asc_file_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/08.Formation_Pressure_Data/FM_PRESS_RAW_RUN7_EWL_1-25_INF_1.ASC"

# Reading as a raw text to inspect the header and data layout
raw_data = spark.read.text(asc_file_path)

print("--- ASC File Content Preview ---")
raw_data.show(truncate=False)

In [0]:
# Attempting to read it as a delimited file if it looks like a table
try:
    # Often these files use multiple spaces as delimiters
    df_pressure = spark.read.option("header", "true").option("inferSchema", "true").csv(asc_file_path)
    
    print("Structure of the Pressure Data:")
    df_pressure.printSchema()
    
    print("Data Preview:")
    display(df_pressure.limit(10))
except Exception as e:
    print(f"Simple CSV read failed, likely due to complex header: {e}")

In [0]:
from pyspark.sql.functions import regexp_extract, col, lit, split, trim

# Load the raw text
raw_asc_df = spark.read.text(asc_file_path)

# 1. Extract Global Metadata using Regex
# We take the value after the colon
well_id = raw_asc_df.filter(col("value").contains("WELL         :")).select(regexp_extract(col("value"), r":\s+(.*)", 1)).collect()[0][0]
rig_name = raw_asc_df.filter(col("value").contains("RIG          :")).select(regexp_extract(col("value"), r":\s+(.*)", 1)).collect()[0][0]

# 2. Extract Table Rows
# We look for lines starting with "8 1/2" (the hole section)
table_data_df = raw_asc_df.filter(col("value").contains("8 1/2\""))

# 3. Parse fixed-width/multi-space columns
# Columns: Section, Service, Company, Pass, Interval, Date, Run, DLIS, OriginalName
parsed_metadata_df = table_data_df.select(
    lit(well_id).alias("well_id"),
    lit(rig_name).alias("rig_name"),
    regexp_extract(col("value"), r"PRETEST @ ([\d.]+)", 1).alias("test_depth_m"),
    regexp_extract(col("value"), r"(\d{2}-[A-Z]{3}-\d{4})", 1).alias("test_date"),
    # Extracting the original DLIS file name from the end of the string
    regexp_extract(col("value"), r"(\S+\.dlis)$", 1).alias("source_dlis_file")
)

display(parsed_metadata_df)

In [0]:
# Assuming your SAP data was saved in the previous step
sap_path = "abfss://data@stshellcrude.dfs.core.windows.net/landing/sap_opex_data/"
sap_df = spark.read.option("header", "true").csv(sap_path)

# Final Business Insights Table
gold_well_analysis = parsed_metadata_df.join(sap_df, on="well_id", how="inner") \
    .select(
        "well_id",
        "well_name",
        "rig_name",
        "test_depth_m",
        "daily_opex",
        "operational_status",
        "test_date"
    )

# Save as a managed Delta table in Unity Catalog
gold_well_analysis.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("crude_ops_catalog.gold.well_test_economics")

display(spark.table("crude_ops_catalog.gold.well_test_economics"))

In [0]:
# Let's search for processed data instead of raw logs
search_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/"

def find_insights(path):
    files = dbutils.fs.ls(path)
    for f in files:
        if f.isDir():
            find_insights(f.path)
        elif f.name.endswith(".csv") or f.name.endswith(".parquet"):
            print(f"Found Insight Candidate: {f.path} ({f.size / 1024 / 1024:.2f} MB)")

find_insights(search_path)

In [0]:
# Scanning the LWD folder for time-series or high-resolution depth data
lwd_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/05.LWD_Log_data/"

files = dbutils.fs.ls(lwd_path)
for f in files:
    # We are looking for larger files or specific keywords like 'Time', 'Log', 'Daily'
    print(f"File: {f.name} | Size: {f.size / 1024 / 1024:.2f} MB")

In [0]:
# Recursive search for any tabular data in the dataset
root_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/"

def scan_for_tables(path):
    items = dbutils.fs.ls(path)
    for item in items:
        if item.isDir():
            scan_for_tables(item.path)
        elif item.name.lower().endswith(('.csv', '.parquet', '.txt')):
            print(f"FOUND TABLE: {item.path} | Size: {item.size / 1024 / 1024:.2f} MB")

scan_for_tables(root_path)

In [0]:
# Select one of the LAS files we found
las_file_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/05.LWD_Log_data/WL_RAW_BHPR-GR-MECH_TIME_MWD_1.LAS"

# Read as text to see the structure and column names
raw_las_df = spark.read.text(las_file_path)

# Show the header (usually metadata about columns is in the first 50-100 rows)
print("--- LAS File Header & Data Preview ---")
raw_las_df.limit(100).show(truncate=False)